In [89]:
import math
import random

In [ ]:
import os
import requests
import re
from bs4 import BeautifulSoup
import pandas as pd

# ! pip install lxml

profilepages = ["https://fbref.com/en/comps/9/Premier-League-Stats", 
                "https://fbref.com/en/comps/12/La-Liga-Stats", 
                "https://fbref.com/en/comps/11/Serie-A-Stats", 
                "https://fbref.com/en/comps/20/Bundesliga-Stats", 
                "https://fbref.com/en/comps/13/Ligue-1-Stats"]

attrsid = ["results2024-202591_overall", 
           "results2024-2025121_overall", 
           "results2024-2025111_overall", 
           "results2024-2025201_overall", 
           "results2024-2025131_overall"]

team_df = pd.DataFrame()

for i in range(len(profilepages)):
    tables = pd.read_html(profilepages[i], attrs={"id": attrsid[i]})[0]
    selected_columns = tables[['Squad', 'MP', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts', 'Last 5', 'Attendance']]
    team_df = pd.concat([team_df, selected_columns], ignore_index=True)

print(type(team_df))

In [31]:
teamURLpullpages = ["https://www.transfermarkt.us/premier-league/tabelle/wettbewerb/GB1", 
                    "https://www.transfermarkt.us/laliga/tabelle/wettbewerb/ES1",
                    "https://www.transfermarkt.us/serie-a/tabelle/wettbewerb/IT1",
                    "https://www.transfermarkt.us/bundesliga/tabelle/wettbewerb/L1",
                    "https://www.transfermarkt.us/ligue-1/tabelle/wettbewerb/FR1"]

headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/47.0.2526.106 Safari/537.36'}

links = []

for i in range(len(teamURLpullpages)):
    page = teamURLpullpages[i]
    pageTree = requests.get(page, headers = headers)
    pageSoup = BeautifulSoup(pageTree.content, 'html.parser')
    Table = pageSoup.find("div", {"class":"responsive-table"}).find_all("td", {"class": "no-border-links hauptlink"}) 
    pattern = r'href="(/[^"]+)"'
    for i in range(len(Table)):
        Final = re.search(pattern, str(Table[i]))
        url = Final.group(1)
        editor = url.split('/')
        editor.pop(6)
        editor.pop(5)
        editor[2] = 'startseite'
        editor.pop(0)
        url_new = 'https://www.transfermarkt.us/'+'/'.join(editor)
        links.append(url_new)

In [108]:
team_reader = pd.read_csv("Football-Data/Team_List.csv")
team_reader['Team Link'] = links
downloads_folder = os.path.expanduser("~/Downloads/intermediatepython-finalproj-kireetijosyula41/Football-Data")
file_path = os.path.join(downloads_folder, "Team_List.csv")
team_reader.to_csv(file_path, index=False)

In [93]:
team_df['Squad'] = TeamNames
team_df['Squad size'] = TeamSize
team_df['Average age'] = AvgAge 
team_df['Foreign Players'] = Foreigners 
team_df['National Team Players'] = Nationals
team_df['Stadium Name'] = StadiumName
team_df['Stadium Capacity'] = StadiumCapacity
team_df['Team Logo Link'] = TeamLogo
team_df['League Logo Link'] = LeagueLogo

for i in range(len(team_df['Attendance'])):
    if math.isnan(team_df['Attendance'][i]):
        team_df['Attendance'][i] = random.randint(1000, int(team_df['Stadium Capacity'][i]))

print(team_df['Attendance'])

0     60265.0
1     60316.0
2     39570.0
3     33266.0
4     52747.0
       ...   
91    13412.0
92    28256.0
93    20783.0
94    31256.0
95    14233.0
Name: Attendance, Length: 96, dtype: float64


In [141]:
shuffled_prem = team_reader['Squad'][0:20].sample(frac=1).reset_index(drop=True)
shuffled_laliga = team_reader['Squad'][20:40].sample(frac=1).reset_index(drop=True)
shuffled_seriea = team_reader['Squad'][40:60].sample(frac=1).reset_index(drop=True)
shuffled_bundesliga = team_reader['Squad'][60:78].sample(frac=1).reset_index(drop=True)
shuffled_ligueone = team_reader['Squad'][78:96].sample(frac=1).reset_index(drop=True)

pairs = [(shuffled_prem[i], shuffled_prem[i + 1]) for i in range(0, len(shuffled_prem), 2)]
matchups = []
for team1, team2 in pairs:
    matchups.append({'Team': team1, 'Opponent': team2})
    matchups.append({'Team': team2, 'Opponent': team1})

pairs2 = [(shuffled_laliga[i], shuffled_laliga[i + 1]) for i in range(0, len(shuffled_laliga), 2)]
for team1, team2 in pairs2:
    matchups.append({'Team': team1, 'Opponent': team2})
    matchups.append({'Team': team2, 'Opponent': team1})

pairs3 = [(shuffled_seriea[i], shuffled_seriea[i + 1]) for i in range(0, len(shuffled_seriea), 2)]
for team1, team2 in pairs3:
    matchups.append({'Team': team1, 'Opponent': team2})
    matchups.append({'Team': team2, 'Opponent': team1})


pairs4 = [(shuffled_bundesliga[i], shuffled_bundesliga[i + 1]) for i in range(0, len(shuffled_bundesliga), 2)]
for team1, team2 in pairs4:
    matchups.append({'Team': team1, 'Opponent': team2})
    matchups.append({'Team': team2, 'Opponent': team1})

pairs5 = [(shuffled_ligueone[i], shuffled_ligueone[i + 1]) for i in range(0, len(shuffled_bundesliga), 2)]
for team1, team2 in pairs5:
    matchups.append({'Team': team1, 'Opponent': team2})
    matchups.append({'Team': team2, 'Opponent': team1})

matchups_df = pd.DataFrame(matchups)

print(matchups_df)

                       Team                Opponent
0                Chelsea FC  Brighton & Hove Albion
1    Brighton & Hove Albion              Chelsea FC
2            Crystal Palace        Newcastle United
3          Newcastle United          Crystal Palace
4   Wolverhampton Wanderers               Fulham FC
..                      ...                     ...
91           Olympique Lyon              LOSC Lille
92              Stade Reims         Montpellier HSC
93          Montpellier HSC             Stade Reims
94              Le Havre AC    RC Strasbourg Alsace
95     RC Strasbourg Alsace             Le Havre AC

[96 rows x 2 columns]


In [142]:
team_df = team_reader
team_df = team_df.drop('Opponents', axis=1)
mapped_df = team_df.merge(matchups_df.rename(columns={'Team': 'Squad'}), on='Squad', how='left')
print(mapped_df)

                     Squad  MP   W  D  L  GF  GA  GD  Pts     Last 5  ...  \
0             Liverpool FC  13  11  1  1  26   8  18   34  D W W W W  ...   
1               Arsenal FC  13   7  4  2  26  14  12   25  D L D W W  ...   
2               Chelsea FC  13   7  4  2  26  14  12   25  W D D W W  ...   
3   Brighton & Hove Albion  13   6  5  2  22  17   5   23  D L W W D  ...   
4          Manchester City  13   7  2  4  22  19   3   23  W L L L L  ...   
..                     ...  ..  .. .. ..  ..  ..  ..  ...        ...  ...   
91              Angers SCO  13   3  4  6  14  21  -7   13  W W L L W  ...   
92        AS Saint-Étienne  13   4  1  8  11  30 -19   13  L W L W L  ...   
93             Le Havre AC  13   4  0  9  10  24 -14   12  L W L W L  ...   
94               FC Nantes  13   2  5  6  15  20  -5   11  L L L L D  ...   
95         Montpellier HSC  13   2  2  9  13  34 -21    8  L L W L D  ...   

    Squad size  Average age  Foreign Players  National Team Players  \
0   

In [144]:
downloads_folder = os.path.expanduser("~/Downloads/intermediatepython-finalproj-kireetijosyula41/Football-Data")
file_path = os.path.join(downloads_folder, "Team_List.csv")
mapped_df.to_csv(file_path, index=False)